# Prophet de Meta
## El puente hacia los modelos modernos

> **Fase 3 — Video 13** | Modelado y Pronóstico
> Dataset: Histórico de Ventas Sintético

---

### De ecuaciones a componentes + regresores

ARIMA (Video 12) piensa en términos de autocorrelación y diferenciación. **Prophet** (de Meta) cambia el
chip: modela la serie como una **suma de componentes interpretables**

$$y(t) = g(t) + s(t) + h(t) + \beta\,x(t) + \varepsilon_t$$

tendencia `g`, estacionalidad `s`, festivos `h` y regresores externos `x`. Esa filosofía de
*"componentes + regresores"* es exactamente la que prepara el salto a los modelos de ML de los próximos
videos. Además Prophet es **fácil**: pocas líneas, festivos nativos, intervalos de incertidumbre y una
descomposición visual preciosa.

La pregunta honesta, la de siempre: **¿le gana al benchmark?** Lo pondremos a prueba.

### Ruta del notebook
1. Librerías y carga (con el arreglo de Prophet en Windows)
2. ¿Por qué Prophet? La filosofía de componentes
3. Un pronóstico en pocas líneas
4. La joya de Prophet: la descomposición de componentes
5. Festivos nativos y regresores externos
6. Veredicto vs. el benchmark
7. Cuándo (y cuándo no) usar Prophet
8. Conclusiones y puente al Video 14

---
## 1. Librerías y carga

> **Nota de instalación.** Prophet usa el backend Stan (`cmdstanpy`). En Windows con idioma español, un
> bug conocido rompe la carga del backend al no encontrar `tbb.dll`. Lo resolvemos agregando al `PATH` la
> carpeta de `tbb.dll` que Prophet ya trae empaquetada — así el notebook corre sin instalar nada extra.

In [ ]:
import warnings, logging, os, sys
warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').setLevel(logging.CRITICAL)

# --- Arreglo del backend de Prophet en Windows (tbb.dll al PATH) ---
import prophet
_tbb = os.path.join(os.path.dirname(prophet.__file__), 'stan_model',
                    'cmdstan-2.37.0', 'stan', 'lib', 'stan_math', 'lib', 'tbb')
if os.path.isdir(_tbb):
    os.environ['PATH'] = _tbb + os.pathsep + os.environ.get('PATH', '')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from prophet import Prophet
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
unit_fmt = FuncFormatter(lambda x, _: f'{x/1e3:.0f}k')
print('✅ Librerías cargadas (Prophet listo)')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
sys.path.insert(0, str(csv_path.parents[1]))
from src.metrics import wape, mase
from src.events import SPECIAL_EVENTS
from src.calendar_mx import MX_HOLIDAYS

df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')
g = df.groupby(pd.Grouper(key='date', freq='W-MON'))
y     = g['units_sold'].sum().iloc[1:-1]
promo = g['discount'].mean().reindex(y.index)

H, m = 52, 52
train, test = y.iloc[:-H], y.iloc[-H:]
snaive = pd.Series(train.iloc[-m:].values[:H], index=test.index)

# Prophet espera columnas ds / y
dfp = pd.DataFrame({'ds': y.index, 'y': y.values, 'promo': promo.values})
train_p = dfp.iloc[:-H].copy()

print(f'✅ {len(y)} semanas | holdout {H} | listón (Seasonal Naive) WAPE {wape(test, snaive):.1%}')

---
## 2. ¿Por qué Prophet? La filosofía de componentes

| | ARIMA / SARIMAX | Prophet |
|---|---|---|
| Enfoque | Autocorrelación + diferenciación | Suma de componentes interpretables |
| Estacionalidad | Fourier manual / SARIMA | Automática (Fourier interno) |
| Festivos | Exógenas que construyes tú | **Nativos**, con ventanas |
| Tendencia | Diferenciación | Lineal a trozos con *changepoints* |
| Incertidumbre | Intervalos de confianza | Intervalos por simulación |
| Curva de aprendizaje | Alta | **Baja** |

Prophet no exige estacionariedad ni leer ACF/PACF: le das `ds` (fecha) e `y` (valor) y decide el resto.
Esa comodidad es su gran atractivo — y el puente conceptual hacia "features + modelo" del ML.

---
## 3. Un pronóstico en pocas líneas

La demanda es **multiplicativa** (los picos crecen con el nivel, Video 4), así que usamos
`seasonality_mode='multiplicative'`. Nota lo corto que es el código.

In [ ]:
m_base = Prophet(yearly_seasonality=10, weekly_seasonality=False, daily_seasonality=False,
                 seasonality_mode='multiplicative')
m_base.fit(train_p)

future = dfp[['ds']].copy()
fc_base = m_base.predict(future)
pred_base = pd.Series(fc_base.set_index('ds')['yhat'].values[-H:], index=test.index)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train.index[-104:], train.values[-104:], color=BLUE, linewidth=1, alpha=0.6, label='train')
ax.plot(test.index, test.values, color='black', linewidth=2.5, label='REAL')
ax.plot(test.index, pred_base.values, color=PURPLE, linewidth=1.8, label='Prophet')
# Banda de incertidumbre
low = fc_base.set_index('ds')['yhat_lower'].values[-H:]
up  = fc_base.set_index('ds')['yhat_upper'].values[-H:]
ax.fill_between(test.index, low, up, color=PURPLE, alpha=0.15, label='intervalo 80%')
ax.axvline(train.index[-1], color='black', linestyle='--', linewidth=1)
ax.yaxis.set_major_formatter(unit_fmt)
ax.set_title('Prophet: pronóstico con intervalo de incertidumbre', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', ncol=2)
plt.tight_layout()
plt.show()

print(f'Prophet base: WAPE {wape(test, pred_base):.3f} | MASE {mase(test, pred_base, train, m):.2f}')

---
## 4. La joya de Prophet: la descomposición de componentes

Su método `plot_components` separa el pronóstico en **tendencia + estacionalidad + festivos**. Es la mejor
herramienta de *comunicación* de Prophet: le explicas a un stakeholder no técnico "esto es la tendencia,
esto la temporada" sin una sola ecuación.

In [ ]:
fig = m_base.plot_components(fc_base)
fig.set_size_inches(11, 7)
plt.tight_layout()
plt.show()
print('Tendencia creciente + un patrón anual con su pico de fin de año: Prophet lo aísla automáticamente.')

---
## 5. Festivos nativos y regresores externos

Aquí Prophet luce: pásale un DataFrame de festivos (con ventanas de anticipación) y se acabó. Añadimos los
eventos comerciales y los festivos mexicanos (Video 8) y un regresor de **promoción** (Video 10).

In [ ]:
years = range(y.index.year.min(), y.index.year.max() + 1)
rows = []
for e in SPECIAL_EVENTS:
    for yr in years:
        rows.append({'holiday': e.name, 'ds': pd.Timestamp(yr, e.month, e.day_start),
                     'lower_window': -3, 'upper_window': 3})
for h in MX_HOLIDAYS:
    for yr in years:
        rows.append({'holiday': h.name, 'ds': pd.Timestamp(yr, h.month, h.day),
                     'lower_window': -2, 'upper_window': 1})
holidays = pd.DataFrame(rows)

def fit_prophet(use_holidays, use_promo):
    mp = Prophet(yearly_seasonality=10, weekly_seasonality=False, daily_seasonality=False,
                 seasonality_mode='multiplicative',
                 holidays=holidays if use_holidays else None)
    if use_promo:
        mp.add_regressor('promo')
    mp.fit(train_p)
    fut = dfp[['ds', 'promo']].copy()
    pred = pd.Series(mp.predict(fut).set_index('ds')['yhat'].values[-H:], index=test.index)
    return pred

pred_hol   = fit_prophet(True, False)
pred_full  = fit_prophet(True, True)

for name, p in [('Prophet base', pred_base), ('+ festivos', pred_hol), ('+ festivos + promo', pred_full)]:
    print(f'  {name:<22} WAPE {wape(test, p):.3f} | MASE {mase(test, p, train, m):.2f}')
print('\nAgregar festivos/regresores es trivial en Prophet — aunque aquí (pocas ocurrencias por')
print('festivo, 4 años) no mejoran: sus coeficientes se estiman con mucho ruido. Fácil ≠ siempre mejor.')

---
## 6. Veredicto vs. el benchmark

Comparamos Prophet contra el listón (Seasonal Naive) y el campeón del Video 12 (SARIMAX).

In [ ]:
final = pd.DataFrame([
    {'modelo': 'Seasonal Naive (benchmark)', 'WAPE': wape(test, snaive)},
    {'modelo': 'Prophet (mejor variante)',   'WAPE': min(wape(test, pred_base), wape(test, pred_hol), wape(test, pred_full))},
    {'modelo': 'SARIMAX (Video 12)',         'WAPE': 0.049},
]).set_index('modelo')
final['MASE'] = [mase(test, snaive, train, m),
                 min(mase(test, pred_base, train, m), mase(test, pred_hol, train, m), mase(test, pred_full, train, m)),
                 0.56]

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = [GREEN if v < 1 else RED for v in final['MASE']]
ax.barh(final.index[::-1], final['MASE'][::-1], color=colors[::-1])
ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='listón (seasonal naive)')
ax.set_title('MASE por modelo (< 1 le gana al benchmark)', fontsize=13, fontweight='bold')
ax.legend()
for i, v in enumerate(final['MASE'][::-1]):
    ax.text(v, i, f' {v:.2f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(final.assign(WAPE=lambda d: (d.WAPE * 100).round(1), MASE=lambda d: d.MASE.round(2)).to_string())
print('\n→ Honestidad: Prophet NO le gana al Seasonal Naive ni al SARIMAX en esta serie (MASE > 1).')
print('  Es cómodo e interpretable, pero la comodidad no es precisión. Hay que medirlo, como todo.')

---
## 7. Cuándo (y cuándo no) usar Prophet

Que no gane en precisión **no lo hace inútil** — resuelve un problema distinto:

| Prophet brilla cuando… | Prophet NO es la mejor opción cuando… |
|---|---|
| Necesitas resultados rápidos e interpretables | Buscas exprimir el último punto de precisión |
| Tienes muchas series y poco tiempo por serie | Tienes una serie clave que justifica un modelo a medida |
| Los festivos/eventos son protagonistas | La dinámica es autoregresiva fina (ahí gana ARIMA) |
| Debes explicarle el forecast a negocio | — |

Prophet es un **excelente segundo baseline**: mejor que el naive, honesto con la incertidumbre y clarísimo
para comunicar. Solo recuerda compararlo — como hicimos — contra alternativas.

---
## 8. Conclusiones y puente al Video 14

### Las reglas que te llevas

1. **Prophet piensa en componentes** (tendencia + estacionalidad + festivos + regresores): el puente
   conceptual hacia el ML.
2. **Su fortaleza es la facilidad y la interpretabilidad**, no la precisión máxima. `plot_components` es oro
   para comunicar.
3. **Festivos y regresores nativos**, en pocas líneas — aunque agregarlos no siempre mejora (mídelo).
4. **Comodidad ≠ precisión.** Aquí Prophet no superó el benchmark; un buen modelo se demuestra, no se asume.
5. **Úsalo como baseline fuerte y a escala**, y reserva los modelos a medida para tus SKUs clave.

### El hueco que ni ARIMA ni Prophet cubren

Todos estos modelos asumen demanda **continua**. Pero en el Video 7 vimos que ~30% del catálogo son
**slow-movers intermitentes** (llenos de ceros), donde ARIMA, Prophet y ML colapsan. Necesitan su propia
familia de métodos.

---

### Próximo video
**Video 14 — Demanda intermitente: Croston, TSB y ADIDA para SKUs lentos**
El video que casi nadie hace y que te posiciona como especialista de Supply Chain.